In [1]:
from torch import nn
import torch  

In [2]:
class Linear(nn.Module):
    def __init__(self, 
                 in_features: int, 
                 out_features: int, 
                 device: torch.device | None=None, 
                 dtype: torch.dtype | None=None):
        super().__init__()
        self.in_features = in_features 
        self.out_features = out_features 
        self.device = device 
        self.dtype = dtype 
        self.W = nn.Parameter(torch.empty((self.in_features, self.out_features), device=self.device, dtype=self.dtype)) #一定要用Parameter注册，否则无法进入优化器
        std = (2 / (self.in_features + self.out_features)) ** 0.5
        torch.nn.init.trunc_normal_(self.W, mean = 0, std = std, a = -3 * std, b = 3 * std) 
    def forward(self, x : torch.Tensor) -> torch.Tensor:
        return x @ self.W


In [5]:
linear = Linear(5, 5)
linear.state_dict()

OrderedDict([('W',
              tensor([[-0.1989, -0.8137, -0.0861,  0.0474,  0.2733],
                      [-0.1565,  0.2870, -0.1366,  0.0682,  0.0411],
                      [-0.3227, -0.2781,  0.5384,  0.1312,  0.1690],
                      [ 0.2614, -0.0500, -0.0088,  0.6910,  0.4543],
                      [-0.7494, -0.0687, -0.4088, -1.3039,  1.0069]]))])

In [3]:
class EmbeddingLayer(nn.Module):
    def __init__(self, 
                num_embeddings: int,
                embedding_dim: int, 
                device: torch.device | None = None, 
                dtype: torch.dtype | None = None):
        super().__init__()
        self.num_embeddings = num_embeddings
        self.embedding_dim = embedding_dim 
        self.device = device 
        self.dtype = dtype 

        self.embedding_matrix = nn.Parameter(torch.empty((self.num_embeddings, self.embedding_dim), device = self.device, dtype = self.dtype)) 
        torch.nn.init.trunc_normal_(self.embedding_matrix, mean = 0, std = 1, a= -3, b = 3) 

    def forward(self, token_ids : torch.Tensor) -> torch.Tensor:
        return self.embedding_matrix[token_ids]


In [7]:
embedding_layer = EmbeddingLayer(5, 5)
embedding_layer.state_dict()
token_ids = torch.LongTensor([[1, 2, 3,4], [0, 1, 2, 3]])
embed = embedding_layer(token_ids)


In [8]:
linear(embed)

tensor([[[ 1.0456, -0.0137, -0.3607,  0.6321, -0.0534],
         [ 0.7487,  0.1012,  1.1395,  0.3003, -1.1506],
         [ 0.1575,  1.6592,  1.0473, -0.4935, -0.8414],
         [-0.3835, -0.9417, -0.0250,  0.0423,  0.7973]],

        [[-1.9240,  0.7736, -0.5460, -2.3909,  1.8146],
         [ 1.0456, -0.0137, -0.3607,  0.6321, -0.0534],
         [ 0.7487,  0.1012,  1.1395,  0.3003, -1.1506],
         [ 0.1575,  1.6592,  1.0473, -0.4935, -0.8414]]],
       grad_fn=<UnsafeViewBackward0>)

In [4]:
class RMSNorm(nn.Module):
    def __init__(self, 
                d_model: int, 
                eps: float = 1e-5, 
                device=None, 
                dtype=None):
        super().__init__()
        self.d_model = d_model
        self.eps = eps 
        self.device = device 
        self.dtype = dtype 

        self.gamma = nn.Parameter(torch.ones(self.d_model, device = self.device, dtype = self.dtype)) 

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        input = x.to(torch.float32) 
        rms_mean = torch.sqrt(torch.square(input).mean(dim = -1, keepdim = True) + self.eps)
        out = input / rms_mean * self.gamma 
        return out.to(x.dtype)

In [14]:
rms = RMSNorm(5) 
rms_output = rms(embed) 
rms_output

tensor([[[-0.1580, -1.5643, -1.2333,  0.9861, -0.1856],
         [-0.8870, -1.9046,  0.5388, -0.3429, -0.4215],
         [-1.8984, -0.7410,  0.8304, -0.3960,  0.0212],
         [ 1.9046, -0.0367,  0.6134,  0.8520,  0.5183]],

        [[-0.4847,  1.5456,  0.6150, -0.1869,  1.4011],
         [-0.1580, -1.5643, -1.2333,  0.9861, -0.1856],
         [-0.8870, -1.9046,  0.5388, -0.3429, -0.4215],
         [-1.8984, -0.7410,  0.8304, -0.3960,  0.0212]]],
       grad_fn=<MulBackward0>)

In [16]:
torch.std(rms_output, dim = -1, keepdim=True)

tensor([[[1.0088],
         [0.8915],
         [1.0058],
         [0.7129]],

        [[0.9123],
         [1.0088],
         [0.8915],
         [1.0058]]], grad_fn=<StdBackward0>)

In [5]:
class SwiGlu(nn.Module):
    def __init__(self, 
                d_model: int, 
                d_ff: int, 
                device: torch.device | None = None, 
                dtype: torch.dtype | None = None):
        super().__init__()
        self.d_model = d_model 
        self.d_ff = d_ff 
        self.device = device 
        self.dtype = dtype 

        self.W1 = Linear(self.d_model, self.d_ff, self.device, self.dtype) 
        self.W2 = Linear(self.d_ff, self.d_model, self.device, self.dtype)
        self.W3 = Linear(self.d_model, self.d_ff, self.device, self.dtype)  

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        w1_out = self.W1(x)
        silu = w1_out / ( 1 + torch.exp(- w1_out)) 
        swi_glu = silu * self.W3(x) 
        return self.W2(swi_glu)

In [30]:
swi_glu = SwiGlu(48, 128)
x =torch.randn((32, 48))
swi_glu(x)

tensor([[-1.2476e-01, -2.4188e-02, -8.1176e-02,  ..., -9.7308e-02,
         -2.2304e-01,  5.7152e-01],
        [ 4.9230e-03,  5.0876e-02, -1.8447e-01,  ...,  3.7787e-01,
         -1.7848e-01,  2.3582e-01],
        [ 2.8014e-01, -3.3484e-01, -6.7001e-01,  ...,  2.1584e-04,
          2.7793e-01,  1.6253e-01],
        ...,
        [ 1.3954e-01, -1.2370e-02, -1.6736e-01,  ..., -5.5309e-01,
         -5.9699e-02, -1.5117e-02],
        [-1.8121e-01, -2.6539e-01,  9.8254e-02,  ...,  5.5400e-01,
         -8.8580e-01, -1.8220e-01],
        [-2.1664e-02,  3.8804e-01,  1.1557e-01,  ...,  4.9262e-01,
          2.2055e-01, -2.6011e-01]], grad_fn=<MmBackward0>)

In [6]:
class RoPE(nn.Module):
    def __init__(self, 
                 theta: float, 
                 d_k: int, 
                 max_seq_len: int, 
                 device: torch.device | None =None):
        super().__init__() #这里一定要先init,不然无法注册buffer
        self.theta = theta 
        self.d_k = d_k 
        self.max_seq_len = max_seq_len 
        self.device = device 

        pos = torch.arange(0, self.max_seq_len)
        d = d_k / 2 
        freq = self.theta ** ( - torch.arange(0, d) / d) 
        rote_matrix = torch.outer(pos, freq)
        self.register_buffer("cos_matrix", torch.cos(rote_matrix), persistent=False)
        self.register_buffer("sin_matrix", torch.sin(rote_matrix), persistent=False) 
        
    def forward(self, 
                x: torch.Tensor, 
                token_positions: torch.Tensor) -> torch.Tensor:
        x_0 = x[..., 0::2]
        x_1 = x[..., 1::2]
        cos_matrix = self.cos_matrix[token_positions]
        sin_matrix = self.sin_matrix[token_positions]
        print(f"cos_matirx shape is {cos_matrix.shape}, x_0 matrix shape is {x_0.shape}")

        output_0 = x_0 * cos_matrix - x_1 * sin_matrix 
        output_1 = x_0 * sin_matrix + x_1 * cos_matrix 
        return torch.stack([output_0, output_1], dim=-1).flatten(-2) #注意stack + flatten实现交错堆叠

In [17]:
rope = RoPE(1000, 5, 10)
x = torch.randn((3, 3, 6))

In [18]:
y = torch.LongTensor([[3, 2, 1], [0, 6, 9], [1, 5, 6]])

In [20]:
rope(x, y)

cos_matirx shape is torch.Size([3, 3, 3]), x_0 matrix shape is torch.Size([3, 3, 3])


tensor([[[ 0.5211,  0.1160, -0.8480,  0.6406, -1.7346,  1.1035],
         [-0.0771,  0.6570, -0.4441,  1.1452,  0.4879, -0.0968],
         [ 0.3993,  1.3277,  1.3685,  0.5640, -0.4156,  0.1545]],

        [[ 0.0181,  0.7128, -0.0108,  1.3858,  1.2488,  0.5976],
         [ 0.6152,  1.2819, -1.8104,  1.2824,  0.8154,  0.0096],
         [-0.6100, -0.5536,  0.3665, -0.6399, -1.8032, -0.6744]],

        [[ 0.3555,  1.1537,  0.6915, -0.1374,  0.3485, -1.4319],
         [-0.0149, -0.5016, -0.2383,  0.2904, -1.4121,  0.1186],
         [-1.7353, -2.4229, -0.1731,  0.7584,  0.3552,  0.8078]]])

In [7]:
def softmax(x: torch.Tensor, dim: int = -1) -> torch.Tensor:
    dim_max = torch.max(x, dim = dim, keepdim=True)[0] #max的结果是一个二元组
    x_exp = torch.exp(x - dim_max)
    norm = x_exp.sum(dim=dim, keepdim=True)
    return x_exp / norm

In [4]:
x = torch.tensor([1.0, 1.0, 1.0, 1.0])
softmax(x)

tensor([0.2500, 0.2500, 0.2500, 0.2500])

In [8]:
class ScaledDotProductAttention(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, 
                querys: torch.Tensor, 
                keys: torch.Tensor, 
                values: torch.Tensor,
                mask: torch.Tensor | None = None) -> torch.Tensor:
        Q = querys
        K = keys.transpose(-2, -1)
        d_k = querys.shape[-1]
        sdpa_score = torch.matmul(Q, K) / (d_k ** 0.5) 
        if mask is not None:
            sdpa_score = sdpa_score.masked_fill(mask == False, -1e9)
        V = values 
        return torch.matmul(softmax(sdpa_score), values)


In [5]:
q = torch.randn([5, 3, 4])
k = torch.randn([5, 3, 4]) 
v = torch.randn([5, 3, 6])
sdpa = ScaledDotProductAttention()
sdpa(q, k, v)

tensor([[[ 0.4085,  0.7755, -0.0785,  0.8876, -0.5448, -1.2816],
         [ 0.1041,  0.7440,  0.0939,  0.7023, -0.6328, -1.3342],
         [-0.7107,  0.5937,  0.0604,  0.3215, -1.1770, -1.7782]],

        [[ 1.5074, -0.8336, -0.1478, -0.3252,  0.8980,  0.2836],
         [ 0.2280, -0.4140,  0.2781, -0.5837,  1.1609, -0.6164],
         [ 1.1091,  0.1754,  0.0687, -0.9404,  0.3578, -0.1700]],

        [[ 0.0624, -1.1068, -0.9038,  0.8960, -0.0101,  0.6637],
         [ 0.3753, -0.7530, -1.0949,  1.3705, -0.1209, -0.0173],
         [ 0.4081, -0.6844, -1.0768,  1.4190, -0.1670, -0.1086]],

        [[ 0.4098,  0.6067, -0.3584,  0.0743,  0.7980,  0.6195],
         [ 0.5073,  0.3537, -0.3097,  0.0703,  1.3363,  0.6778],
         [ 0.6608,  0.4535, -0.3844, -0.1379,  1.4034,  0.5264]],

        [[-0.4429,  0.0374, -0.8712, -1.4420,  1.1598,  0.1642],
         [-0.2805, -0.1265, -0.5445, -1.2249,  1.0726,  0.2133],
         [-0.3314, -0.1921,  0.2163, -0.8929,  0.8006,  0.4138]]])

In [9]:
class MultiheadSelfAttention(nn.Module):
    def __init__(self, 
                 d_model: int, 
                 num_heads: int, 
                 device: torch.device | None = None, 
                 dtype: torch.dtype | None = None):
        super().__init__()
        self.d_model = d_model
        self.num_heads = num_heads
        self.device = device 
        self.dtype = dtype 

        self.Wq = Linear(self.d_model, self.d_model, self.device, self.device)
        self.Wk = Linear(self.d_model, self.d_model, self.device, self.device)
        self.Wv = Linear(self.d_model, self.d_model, self.device, self.device)
        self.Wo = Linear(self.d_model, self.d_model, self.device, self.device)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        seq_len = x.shape[-2]
        q_proj = self.Wq(x) 
        k_proj = self.Wk(x)
        v_proj = self.Wv(x)
        q_heads = q_proj.view(-1, seq_len, self.num_heads, self.d_model // self.num_heads) #注意这里的除号，以及维度推断只能有一个-1
        k_heads = k_proj.view(-1, seq_len, self.num_heads, self.d_model // self.num_heads)
        v_heads = v_proj.view(-1, seq_len, self.num_heads, self.d_model // self.num_heads)

        q_heads = torch.permute(q_heads, (0, 2, 1, 3))
        k_heads = torch.permute(k_heads, (0, 2, 1, 3))
        v_heads = torch.permute(v_heads, (0, 2, 1, 3)) 

        mask = torch.tril(torch.ones(seq_len, seq_len, dtype = torch.bool)) 

        sdpa = ScaledDotProductAttention()
        o_heads = sdpa(q_heads, k_heads, v_heads, mask) 
        o_heads = torch.permute(o_heads, (0, 2, 1, 3)).contiguous() 
        o = o_heads.view(-1, seq_len, self.d_model) #注意这里要写d_model，而不是d_model * num_heads
        output = self.Wo(o)
        return output 

In [21]:
in_feature = torch.randn([6, 10, 16]) 
nhsa = MultiheadSelfAttentiuon(16, 2)
out = nhsa(in_feature)
nhsa.state_dict().keys()

q_proj shape is torch.Size([6, 10, 16])
q_heads shape is torch.Size([6, 10, 2, 8])
q_heads shape is torch.Size([6, 2, 10, 8])
o_heads shape is torch.Size([6, 2, 10, 8])
o_heads shape is torch.Size([6, 10, 2, 8])
o shape is torch.Size([6, 10, 16])


odict_keys(['Wq.W', 'Wk.W', 'Wv.W', 'Wo.W'])

In [10]:
class MultiheadSelfAttentionRoPE(nn.Module):
    def __init__(self, 
                 d_model: int, 
                 num_heads: int, 
                 max_seq_len: int,
                 theta: float, 
                 device: torch.device | None = None, 
                 dtype: torch.dtype | None = None):
        super().__init__()
        self.d_model = d_model
        self.num_heads = num_heads 
        self.max_seq_len = max_seq_len
        self.theta = theta
        self.device = device 
        self.dtype = dtype  

        self.Wq = Linear(self.d_model, self.d_model, self.device, self.device)
        self.Wk = Linear(self.d_model, self.d_model, self.device, self.device)
        self.Wv = Linear(self.d_model, self.d_model, self.device, self.device)
        self.Wo = Linear(self.d_model, self.d_model, self.device, self.device)
        self.rope = RoPE(theta, self.d_model // self.num_heads, max_seq_len, device=self.device)

    def forward(self, x: torch.Tensor, token_positions: torch.Tensor | None = None) -> torch.Tensor:
        seq_len = x.shape[-2]
        q_proj = self.Wq(x) 
        k_proj = self.Wk(x)
        v_proj = self.Wv(x)

        q_heads = q_proj.view(-1, seq_len, self.num_heads, self.d_model // self.num_heads) #注意这里的除号，以及维度推断只能有一个-1
        k_heads = k_proj.view(-1, seq_len, self.num_heads, self.d_model // self.num_heads)
        v_heads = v_proj.view(-1, seq_len, self.num_heads, self.d_model // self.num_heads)

        q_heads = torch.permute(q_heads, (0, 2, 1, 3))
        k_heads = torch.permute(k_heads, (0, 2, 1, 3))
        v_heads = torch.permute(v_heads, (0, 2, 1, 3)) 

        if token_positions is not None:
            rope_heads = token_positions.unsqueeze(dim = 1).expand(q_heads.shape[0], self.num_heads, seq_len)
            q_heads = self.rope(q_heads, rope_heads)
            k_heads = self.rope(k_heads, rope_heads)

        mask = torch.tril(torch.ones(seq_len, seq_len, dtype = torch.bool)) 

        sdpa = ScaledDotProductAttention()
        o_heads = sdpa(q_heads, k_heads, v_heads, mask) 
        o_heads = torch.permute(o_heads, (0, 2, 1, 3)).contiguous() 
        o = o_heads.view(-1, seq_len, self.d_model) #注意这里要写d_model，而不是d_model * num_heads
        output = self.Wo(o)
        return output 

In [11]:
mhsarope = MultiheadSelfAttentionRoPE(32, 2, 100, 1000.0)

In [12]:
in_features = torch.randn([3, 5, 32])
#token_posions = torch.LongTensor([[0, 1, 2, 3, 5], [3, 6, 9, 11, 2], [5,7,8,9,10]])
mhsarope(in_features)

tensor([[[-3.9376e-01,  1.6878e-01,  9.8825e-01, -4.5683e-02,  1.7995e+00,
           1.4296e+00, -5.0476e-01, -9.9856e-01,  1.1748e+00,  3.4549e-01,
           2.3404e+00, -7.5353e-01,  6.0665e-01, -1.8751e+00,  9.5213e-01,
           1.4505e+00, -9.6561e-01,  1.0721e+00,  1.9674e-01, -1.4756e+00,
          -4.7510e-01, -6.8714e-02,  3.5650e-01,  2.1241e+00, -7.7482e-03,
          -1.8614e+00,  7.6827e-01, -1.5919e-01, -1.3324e+00, -3.0959e-01,
          -2.6068e-02, -4.2697e-01],
         [-6.1021e-01,  8.4465e-02,  7.8329e-01,  5.5549e-01,  7.4774e-01,
           1.8124e-01, -3.7145e-01, -9.7793e-01,  3.7583e-01,  2.1709e-01,
           1.5757e+00, -1.0844e-01,  4.9069e-01, -2.3005e-01,  8.0875e-01,
           5.7859e-01, -1.4398e-01,  1.3589e+00,  3.5551e-01, -1.8212e+00,
          -1.4015e+00,  1.5673e-02,  3.9262e-02,  1.5942e+00, -5.1750e-01,
          -1.8736e+00,  5.4710e-01, -4.0334e-01, -1.7022e+00, -6.1837e-01,
           6.7761e-01, -1.0909e+00],
         [-2.1088e-02,  3.

In [ ]:
class Transformer_Block(nn.Module):
    def __init__(self, 
                d_model: int,
                num_heads: int,
                d_ff: int,
                max_seq_len: int,
                theta: float,
                device: torch.device | None = None,
                dtype: torch.dtype | None = None):
        super().__init__()
        self.d_model = d_model 
        self.num_heads = num_heads 
        self.d_ff = d_ff 
        self.max_seq_len = max_seq_len 
        self.theta = theta
        self.device = device 
        self.dtype = dtype 

        self.rms_norm1 = RMSNorm(self.d_model, 0.00001, self.device, self.dtype) 
        self.mhsarope = MultiheadSelfAttentionRoPE(self.d_model, self.num_heads, self.max_seq_len, self.theta, self.device, self.dtype)
        self.rms_norm2 = RMSNorm(self.d_model, 0.00001, self.device, self.dtype)
        self.ffn = SwiGlu(self.d_model, self.d_ff, self.device, self.dtype) 

    def forward(self, x: torch.Tensor) -> torch.Tensor :
        seq_len = x.shape[-2] 
        batch_size = x.shape[-3]
        token_pos = torch.unsqueeze(torch.arrange(seq_len), dim = 0).expand([batch_size, seq_len])
        x_norm1 = self.rms_norm1(x) 
        x_mhsarope = self.mhsarope(x_norm1, token_pos) 
        x_o1 = x_mhsarope + x
        x_norm2 = self.rms_norm2(x_o1) 
        x_ffn = self.ffn(x_norm2) 
        x_o2 = x_ffn + x_o1
        return x_o2
        

In [21]:
transformer_block = Transformer_Block(64, 2, 48, 256, 1000) 
x = torch.randn([6, 10, 64]) 
x_output = transformer_block(x)

In [24]:
transformer_block.state_dict().keys()

odict_keys(['rms_norm1.gamma', 'mhsarope.Wq.W', 'mhsarope.Wk.W', 'mhsarope.Wv.W', 'mhsarope.Wo.W', 'rms_norm2.gamma', 'ffn.W1.W', 'ffn.W2.W', 'ffn.W3.W'])

In [22]:
x_output

tensor([[[-0.4460, -0.0050, -0.6549,  ...,  0.0292, -0.1814,  0.3410],
         [-0.4195,  0.1644, -0.0258,  ..., -0.7719, -0.4949, -0.4250],
         [-1.1918, -0.3409, -0.0317,  ..., -0.8740, -0.2796, -1.0052],
         ...,
         [ 0.5324,  0.5878,  0.1550,  ..., -0.2453, -0.5047, -0.5971],
         [ 0.1922,  0.4445, -0.1007,  ..., -0.0268, -0.4125, -0.3361],
         [ 0.8802,  0.0708, -2.0182,  ..., -1.4039,  0.3341,  0.1875]],

        [[ 0.3444,  0.3410,  0.6717,  ...,  0.2015, -0.0830,  0.0080],
         [ 0.8649, -0.1737,  0.9513,  ...,  0.2448,  0.6130, -0.7144],
         [ 0.6908,  0.3465,  0.8962,  ..., -0.5698,  1.0620, -0.5847],
         ...,
         [-0.4825,  1.2558,  0.9619,  ..., -0.4785,  1.3077, -0.2347],
         [-0.4374,  0.2330, -0.1886,  ..., -1.1604, -0.3502, -0.2424],
         [-0.1165,  0.3501,  0.8848,  ..., -0.0681,  0.3695, -0.3391]],

        [[ 0.0492, -0.8692, -0.1648,  ..., -0.2791,  0.8197, -0.8572],
         [-0.3293,  0.1093, -0.1706,  ..., -0